In [ ]:
# Standard imports
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

from sklearn.impute import SimpleImputer


### Getting our data to be used with ml

1. Split the data into features and labels( X & Y)
2. Filling or disregarding missing values
3. Convering non-numerical values to numerical values (feature encoding)

In [ ]:
heart_disease = pd.read_csv("heart-disease.csv")
heart_disease.head()



In [ ]:
# 1. Split the data into features and labels( X & Y)
x = heart_disease.drop("target", axis=1)
y = heart_disease["target"]

# Split the data into training and test sets
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)


In [ ]:
# 1.1 Make sure it's all numerical
car_sales = pd.read_csv("car-sales-extended.csv")

car_sales, car_sales.dtypes


In [ ]:
# Split into x/y
x = car_sales.drop("Price", axis=1)
y = car_sales["Price"]

# Turn the categories into numbers
categorical_features = ["Make", "Colour", "Doors"]

one_hot = OneHotEncoder()
transformer = ColumnTransformer([("one_hot", one_hot, categorical_features)], remainder="passthrough")

transformed_x = transformer.fit_transform(x)

# x, x.dtypes, y, y.dtypes
transformed_x
pd.DataFrame(transformed_x)

In [ ]:
dummies = pd.get_dummies(car_sales[["Make", "Colour", "Doors"]])
dummies

In [ ]:
# Split into training and test

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

# x_train, x_test, y_train, y_test = train_test_split(dummies, y, test_size=0.2)

In [ ]:
# Build machine leaning model
model = RandomForestRegressor()
model.fit(x_train, y_train)
model.score(x_test, y_test)

### Handling missing values

1. Fill them with some value / imputation
2. Remove the samples with missing data altogether

In [ ]:
car_sales_missing = pd.read_csv("car-sales-extended-missing-data.csv")
car_sales_missing.head(20), car_sales_missing.dtypes

In [ ]:
car_sales_missing.isna().sum()

In [ ]:
# Fill missing data with pandas
car_sales_missing["Odometer (KM)"] = car_sales_missing["Odometer (KM)"].fillna(car_sales_missing["Odometer (KM)"].mean())

# Drop missing rows
car_sales_missing = car_sales_missing.dropna(subset=["Make", "Colour", "Price"])

# Fill missing value with a specified value
car_sales_missing["Doors"] = car_sales_missing["Doors"].fillna(4)

### Using sklearn to fill missing data

In [ ]:
car_sales_missing2 = pd.read_csv("car-sales-extended-missing-data.csv")

# Fill categorical values with 'missing' and numerical values with mean
cat_imputer = SimpleImputer(strategy="constant", fill_value="missing")
door_imputer = SimpleImputer(strategy="constant", fill_value=4)
num_imputer = SimpleImputer(strategy="mean")

# Define coluns
cat_features = ["Make", "Colour"]
door_feature = ["Doors"]
num_features = ["Odometer (KM)"]

# Create an imputer - something that fills missing data
imputer = ColumnTransformer([
    ("cat_imputer", cat_imputer, cat_features),
    ("door_imputer", door_imputer, door_feature),
    ("num_imputer", num_imputer, num_features)
    ])

# Transform data
car_sales_missing2 = imputer.fit_transform(car_sales_missing2)
car_sales_missing2


### Convert non-numerical data to numerical data
1. Integer encoding - use it when the values have ordinal data
```python
        from sklearn.preprocessing import LabelEncoder, OrdinalEncoder
        import pandas as pd

        df = pd.DataFrame({
            'size': ['small', 'large', 'medium', 'small', 'large']
        })

        # OrdinalEncoder lets you define the order explicitly
        enc = OrdinalEncoder(categories=[['small', 'medium', 'large']])
        df['size_encoded'] = enc.fit_transform(df[['size']])

```

2. One-Hot Encoding
```python
        # Using pandas
        import pandas as pd

        df = pd.DataFrame({
            'color': ['red', 'blue', 'green', 'red', 'blue']
        })

        df_encoded = pd.get_dummies(df, columns=['color'], dtype=int)

        # Using sklearn
        enc = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
        enc.fit(train[['color']])

        enc.transform(test[['color']])
```

In [ ]:
# Converting non-numerical values to numerical values using sklearn
from sklearn.preprocessing import OneHotEncoder


enc = OneHotEncoder(handle_unknown="ignore", sparse_output=False).set_output(transform="pandas")

encoded = enc.fit_transform(car_sales_missing[["Make", "Colour"]])
car_sales_cleaned = pd.concat([car_sales_missing, encoded], axis=1).drop(columns = ["Make", "Colour"])

In [ ]:
car_sales_cleaned.dtypes, car_sales_cleaned.isna().sum()

In [ ]:
# Split data into train and testing
x = car_sales_cleaned.drop(columns=["Price"])
y = car_sales_cleaned["Price"]

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2)
x, y

In [ ]:
# Select and train model
model = RandomForestRegressor()
model.fit(x_train, y_train)
model.score(x_test, y_test)